In [1]:
import nbref.objects, nbref.schema
importlib.reload(nbref.objects)
importlib.reload(nbref.schemas)
# importlib.reload(nbref.html)
# importlib.reload(nbref.utils)
from nbref.html import *
import nbref.roles
import nbref.htmls

from nbref.objects import *
from pandas import Series, DataFrame
import referencing, jsonschema_specifications
# referencing.jsonschema.bool = nbref.objects.Boolean


In [2]:
%%

    header = Schema\
```yaml
type: array
parent:
    role: banner
    aria: no
prefixItems:
-   role: link
    tags: [skip]
    default: skip to top
    attrs:
        href: "#TOP"
-   role: link
    tags: [skip]
    default: skip to bottom
    attrs:
        href: "#BOTTOM"
-   role: img
    type: string
    parent:
        role: link
        type: string
        attrs:
            href: "#"
            title: tonyfast
    attrs:
        src: https://avatars.githubusercontent.com/u/4236275?v=4
        style:
            height: 64px;
-   role: button
    default: settings
    on:
        click: event.target.ariaControls.forEach((object)=>object.showModal())
    aria:
        controls: settings:dialog
```

In [3]:
    header().set_name("header").html()

In [4]:
%%

    import jsonschema
    (settings := Schema(
```yaml
$id: https://deathbeds.github.io/nbref/settings
title: settings
type: object
description: update how the document looks, sounds, and feels.
$anchor: settings
properties:
    color:
        title: color settings
        description: change the foreground and background colors
        properties:
            theme: 
                description: update the color theme
                type: string
                default: system
                enum: [light, dark, system]
                $ref: "#/$defs/toggleClassHTML"
            background:
                title: background image
                description: show the background image
                type: boolean
                default: true
            filters:
                title: color filters
                properties: 
                    invert:                        
                        type: boolean
                        $ref: "#/$defs/toggleClassHTML"
                    gray:
                        type: boolean
                        $ref: "#/$defs/toggleClassHTML"

    typography:
        title: typography
        # role: details
        properties: 
            font-size:
                enum: [medium]
            font:
                title: prefer the system font
                default: false
                type: boolean
                $comment: the default font is atkinson hyperlegible
            document:
                title: modify the document layout
                properties:
                    margin:
                        type: number
                        minimum: 0
                        default: 5
                        dependentSchemas:
                            margin-unit: 
                                default: em
                                $ref: "#/$defs/css:length"
                        unit:
                            default: em
                            
                    width:
                        title: document width
                        default: fit
                        oneOf:
                            - type: string
                              enum: [fit, wide]
                              $ref: "#/$defs/toggleClassHTML"
                            - title: custom
                              type: number
                              default: 100
                              aria:
                                controls: HTML
                              "on":
                                change: setStyleProperty(event)
                              data:
                                key: margin
                              dependentSchemas:
                                  unit:
                                    $ref: "#/$defs/css:length"
            custom:
                title: custom styles
                type: array
                additionalItems:
                    properties:
                        style:
                            type: string
                            oneOf:
                            - title: url
                              type: string
                              format: uri
                            - title: css
                              type: string
                              format: textarea
                              contentSchema:
                                  contentMediaType: text/css
                        visible:
                            type: boolean
                            default: true
                                                    
            # spacing:
            #     title: 
            #     properties:
            #         letter:
                        
    # at:
    #     title: assistive technology
    # properties: {}
$defs:
    toggleClassHTML: 
        aria:
            controls: HTML
        "on":
            change: toggleClass(event)
    css:length:
        type: string
        default: "%"
        enum: [em, "%"]
parent: 
    role: dialog
    aria:
        expanded: no
``` 
    ).expand()().set_name("settings")).render_html()

In [5]:
def normalize_notebook(object):
    for cell in object.get("cells", []):
        if not cell.get("id"):
            cell["id"] = str()
        if cell["cell_type"] == "markdown":
            cell["outputs"] = [
                dict(
                    output_type="display_data", data={"text/markdown": cell["source"]}
                )
            ]

    object["cells"] = object["cells"][:5]
    return object

In [6]:
def normalize_source(schema, object, options, **attrs):
    return schema, object.reflect(Schema.String("".join(object)))

In [7]:
options = Schema.Options(schema_patch={
        ("#", "properties", "cells"): lambda *x: print("ding dig"),
        ("#", "properties", "cells", "items"): 
        lambda schema, object, options, **attrs: 
        (Schema(schema, tags=["cell", object["cell_type"]], title=F"{object["cell_type"]} {schema.title()} {object.path[-1] + 1}").pipe(schema.reflect),),
        ("#", "properties", "cells", "items", "properties", "source"): normalize_source
    })

In [8]:

raw_nb = dict(cells=[
    dict(cell_type="raw", outputs=[dict(
        output_type="display_data",
        data={
            "text/uri-list": Path("~/tonyfast/tonyfast/xxv/2025-10-28-stream.ipynb").expanduser().as_uri()
        }
    )])
])

In [9]:
ls /Users/tonyfast/antisocial/making.ipynb

/Users/tonyfast/antisocial/making.ipynb


In [10]:
raw_nb = dict(cells=[
    dict(cell_type="raw",
         source="""file://~/tonyfast/tonyfast/xxv/2025-10-28-stream.ipynb
file:///Users/tonyfast/antisocial/making.ipynb
         """,
    )
])

In [ ]:

Schema.Notebook(raw_nb).html()

In [ ]:
Object.dispatch(raw_nb).set_schema(NOTEBOOK)["cells", 0, "outputs", 0, "data"]

In [12]:
(
    nb := Object.from_file(file:="~/tonyfast/tonyfast/xxv/2025-10-28-stream.ipynb")
        .set_schema(NOTEBOOK)
).set_name("nb").pipe(normalize_notebook).render_html(
    
    Schema.Options(schema_patch={
        ("#", "properties", "cells"): lambda *x: print("ding dig"),
        ("#", "properties", "cells", "items"): 
        lambda schema, object, options, **attrs: 
        (Schema(schema, tags=["cell", object["cell_type"]], title=F"{object["cell_type"]} {schema.title()} {object.path[-1] + 1}").pipe(schema.reflect),),
        ("#", "properties", "cells", "items", "properties", "source"): normalize_source
    },
    el("style", """
    span.type {
        display: none;
    }
    ul.nb, ol.cells, ul.cell, ol.outputs, ul.output, ul.data {
        & > li {
            list-style: none;
        }
    }
    main.nb {
        ul.nb>li:not(.cells),
        a.anchor:not(.cell),
        ul.cell>li:not(.source, .outputs),
        ul.output>li:not(.data),
        ul.data > li> details > summary {
            display: none;
        }
    }
    """[:]),
)

ding dig


In [ ]:
nb["cells"][0]["outputs"][0]["data"].schema.pipe(Schema.role)

In [211]:
tb = sys.last_exc.__traceback__

In [7]:
    for t, l in traceback.walk_tb(tb):
        pass

NameError: name 'tb' is not defined

In [213]:
    t.f_locals

{'self': <li class="oneof data object"><input aria-aria-labelledby="nb:cells:0:outputs:0:data:title" aria-owns="nb:cells:0:outputs:0:data" checked="" class="radio" name="nb:cells:0:outputs:0:data:oneof" type="radio" value="data"></input></li>, 'position': 1, 'new_child': {
    "text/markdown": [
        "# aria live and in complex web applications"
    ]
}, 'BeautifulSoup': <class 'bs4.BeautifulSoup'>}

In [178]:
def trace_objects(tb=None, names=["nbref"]):
    if tb is None:
        tb = sys.last_traceback
    errors = []
    done = []
    for t, l in traceback.walk_tb(tb):
        if "nbref" in t.f_code.co_filename:
            f = [x for x in gc.get_referrers(t.f_code) if callable(x) and getattr(x, "__code__", None) is t.f_code][0]
            sig = inspect.signature(f)
            objects = []
            for param in list(sig.parameters)[1:2]:
                x = t.f_locals[param]
                if isinstance(x, Schema.Object):
                    if tuple(x.path) not in done:
                        errors.append((x.path, x))
                        done.append(tuple(x.path))
    return errors

In [77]:
t.f_back.

{'__name__': 'bs4.element',
 '__doc__': None,
 '__package__': 'bs4',
 '__loader__': <_frozen_importlib_external.SourceFileLoader at 0x10dc659d0>,
 '__spec__': ModuleSpec(name='bs4.element', loader=<_frozen_importlib_external.SourceFileLoader object at 0x10dc659d0>, origin='/Users/tonyfast/.pixi/envs/default/lib/python3.14/site-packages/bs4/element.py'),
 '__file__': '/Users/tonyfast/.pixi/envs/default/lib/python3.14/site-packages/bs4/element.py',
 '__cached__': '/Users/tonyfast/.pixi/envs/default/lib/python3.14/site-packages/bs4/__pycache__/element.cpython-314.pyc',
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-i

In [ ]:
sys.last_exc.__traceback__.tb_next

In [60]:
tb = sys.last_exc.__traceback__
stack = [sys.last_exc.__traceback__]
while stack[-1].tb_next:
    stack.append(stack[-1].tb_next)

In [64]:
s = traceback.extract_stack(tb.tb_frame)[0]

In [ ]:
s.

In [45]:
print(sys.last_exc)
tbs = traceback.extract_tb(sys.last_exc.__traceback__)
for summary in tbs:
    print(summary)

'Dict' object has no attribute 'extract'
<FrameSummary file /Users/tonyfast/.pixi/envs/default/lib/python3.14/site-packages/IPython/core/interactiveshell.py, line 3747 in run_code>
<FrameSummary file /var/folders/px/9_2c75mx2rv60qjp1v7g7wfw0000gn/T/ipykernel_14736/3963168737.py, line 4 in <module>>
<FrameSummary file /Users/tonyfast/.pixi/envs/default/lib/python3.14/site-packages/nbref/objects.py, line 266 in render_html>
<FrameSummary file /Users/tonyfast/.pixi/envs/default/lib/python3.14/site-packages/nbref/objects.py, line 262 in render_bs4>
<FrameSummary file /Users/tonyfast/.pixi/envs/default/lib/python3.14/site-packages/nbref/htmls.py, line 21 in html_render>
<FrameSummary file /Users/tonyfast/.pixi/envs/default/lib/python3.14/site-packages/nbref/htmls.py, line 42 in html_applicator>
<FrameSummary file /Users/tonyfast/.pixi/envs/default/lib/python3.14/site-packages/nbref/htmls.py, line 140 in html_one_of>
<FrameSummary file /Users/tonyfast/.pixi/envs/default/lib/python3.14/site-p

In [49]:
traceback.t(summary)

<FrameSummary file /Users/tonyfast/.pixi/envs/default/lib/python3.14/site-packages/bs4/element.py, line 2160 in _insert>

In [34]:
pipe(
    sys.last_traceback.tb_frame,
    inspect.getouterframes
)

[FrameInfo(frame=<frame at 0x168cfefc0, file '/Users/tonyfast/.pixi/envs/default/lib/python3.14/site-packages/IPython/core/interactiveshell.py', line 3772, code run_code>, filename='/Users/tonyfast/.pixi/envs/default/lib/python3.14/site-packages/IPython/core/interactiveshell.py', lineno=3772, function='run_code', code_context=['        return outflag\n'], index=0, positions=Positions(lineno=3772, end_lineno=3772, col_offset=8, end_col_offset=22))]

# anti = Schema.Object.from_file("/Users/tonyfast/antisocial/index.ipynb")

In [ ]:
anti.set_schema(NOTEBOOK).pipe(
    normalize_notebook
).render_html()

In [ ]:
isinstance(Schema.String("abc"), str)

In [ ]:
nb["cells"].schema.path

In [ ]:
Schema(type="number")().modify(title="numb").render_html()

In [ ]:
Schema()(a=11, b=12).infer_schema().modify(title="numb", description="you bitch").render_html()

In [ ]:
Schema(oneOf=[
    dict(type="number"), dict(type="string")
])

In [ ]:
Schema(dict(oneOf=[
    dict(type="number"), dict(type="string")
])).expand()(1).display()

In [ ]:
Schema(oneOf=)(a=1).infer_schema().modify(title="jawn").render_html()

In [ ]:
sf =Series(Schema.Ref.REGISTRY).apply(
    operator.attrgetter("contents")
).apply(Series)

In [ ]:
sf[sf.index.str.startswith("https://json-schema.org/draft/2020-12")]["properties"].apply(Series).T.style

In [ ]:
    meta = Schema[Schema.from_file(nbref.schema.parent / "meta.yaml").expand()]
    Schema.Ref.register_schema(meta.schema)

In [ ]:
    nb = meta.from_file(nbref.schema.parent / "nb.yaml").expand()

In [ ]:
    nb.property("cells").property(0).property("source")

In [ ]:
    document = Object.from_file(Path("~/tonyfast/tonyfast/xxv/2025-08-27-playwright-esm.ipynb").expanduser()).pipe(nb)

role is the primary verb in the representation of the object in html

In [ ]:
    document["cells", 0, "source"].modify(
        type="string",
    ).display()

In [ ]:
    nb

In [ ]:
    nb

In [ ]:
%%

    import jsonschema
    (settings := meta(
```yaml
$id: https://deathbeds.github.io/nbref/settings
title: settings
type: object
description: update how the document looks, sounds, and feels.
role: region
$anchor: settings
properties:
    color:
        title: color settings
        description: change the foreground and background colors
        properties:
            theme: 
                description: update the color theme
                type: string
                default: system
                enum: [light, dark, system]
                $ref: "#/$defs/toggleClassHTML"
            background:
                title: background image
                description: show the background image
                type: boolean
                default: true
            filters:
                title: color filters
                properties: 
                    invert:                        
                        type: boolean
                        $ref: "#/$defs/toggleClassHTML"
                    gray:
                        type: boolean
                        $ref: "#/$defs/toggleClassHTML"

    typography:
        title: typography
        # role: details
        properties: 
            font-size:
                enum: [medium]
            font:
                title: prefer the system font
                default: false
                type: boolean
                $comment: the default font is atkinson hyperlegible
            document:
                title: modify the document layout
                properties:
                    margin:
                        type: number
                        minimum: 0
                        default: 5
                        dependentSchemas:
                            margin-unit: 
                                default: em
                                $ref: "#/$defs/css:length"
                        unit:
                            default: em
                            
                    width:
                        title: document width
                        default: fit
                        oneOf:
                            - type: string
                              enum: [fit, wide]
                              $ref: "#/$defs/toggleClassHTML"
                            - title: custom
                              type: number
                              default: 100
                              aria:
                                controls: HTML
                              "on":
                                change: setStyleProperty(event)
                              data:
                                key: margin
                              unit:
                                $ref: "#/$defs/css:length"
            custom:
                title: custom styles
                type: array
                role: details
                additionalItems:
                    properties:
                        style:
                            type: string
                            oneOf:
                            - title: url
                              type: string
                              format: uri
                            - title: css
                              type: string
                              format: textarea
                              contentSchema:
                                  contentMediaType: text/css
                        visible:
                            type: boolean
                            default: true
                                                    
            # spacing:
            #     title: 
            #     properties:
            #         letter:
                        
    # at:
    #     title: assistive technology
    # properties: {}
$defs:
    toggleClassHTML: 
        aria:
            controls: HTML
        "on":
            change: toggleClass(event)
    css:length:
        type: string
        default: "%"
        enum: [em, "%"]
``` 
    ))
    state = settings(
```yaml
color:
    theme: system
    background: true
    filters: 
        invert: no
        gray: no
custom: []
typography:
    font-size: medium
    margin: 5
    document:
        width: fit
```
    )

In [ ]:
state.display()

In [ ]:
Ref.register_schema(SETTINGS).crawl()

In [ ]:
data["color"]["filters"]["invert"]

In [ ]:
Ref("https://deathbeds.github.io/nbref/settings").resolve()

In [ ]:
%%

    import jsonschema
    (SETTINGS := Schema[META](
```yaml
$id: https://deathbeds.github.io/nbref/settings
title: settings
type: object
description: update how the document looks, sounds, and feels.
role: region
$anchor: settings
properties:
    color:
        title: color settings
        description: change the foreground and background colors
        properties:
            theme: 
                description: update the color theme
                type: string
                default: system
                enum: [light, dark, system]
                $ref: "#/$defs/toggleClassHTML"
            background:
                title: background image
                description: show the background image
                type: boolean
                default: true
            filters:
                title: color filters
                properties: 
                    invert:                        
                        type: boolean
                        $ref: "#/$defs/toggleClassHTML"
                    gray:
                        type: boolean
                        $ref: "#/$defs/toggleClassHTML"

    typography:
        title: typography
        # role: details
        properties: 
            font-size:
                enum: [medium]
            font:
                title: prefer the system font
                default: false
                type: boolean
                $comment: the default font is atkinson hyperlegible
            document:
                title: modify the document layout
                properties:
                    margin:
                        type: number
                        minimum: 0
                        default: 5
                        unit:
                            default: em
                            $ref: "#/$defs/css:length"
                    width:
                        title: document width
                        default: fit
                        oneOf:
                            - type: string
                              enum: [fit, wide]
                              $ref: "#/$defs/toggleClassHTML"
                            - title: custom
                              type: number
                              default: 100
                              aria:
                                controls: HTML
                              "on":
                                change: setStyleProperty(event)
                              data:
                                key: margin
                              unit:
                                $ref: "#/$defs/css:length"
            custom:
                title: custom styles
                type: array
                role: details
                additionalItems:
                    properties:
                        style:
                            type: string
                            oneOf:
                            - title: url
                              type: string
                              format: uri
                            - title: css
                              type: string
                              format: textarea
                              contentSchema:
                                  contentMediaType: text/css
                        visible:
                            type: boolean
                            default: true
                                                    
            # spacing:
            #     title: 
            #     properties:
            #         letter:
                        
    # at:
    #     title: assistive technology
    # properties: {}
$defs:
    toggleClassHTML: 
        aria:
            controls: HTML
        "on":
            change: toggleClass(event)
    css:length:
        type: string
        default: "%"
        enum: [em, "%"]
``` 
    ))
    settings = data = Dict[SETTINGS](
```yaml
color:
    theme: system
    background: true
    filters: 
        invert: no
        gray: no
custom: []
typography:
    font-size: medium
    margin: 5
    document:
        width: fit
```
    )

In [ ]:
data.schema.schema

In [ ]:
data["typography"].schema

In [ ]:
00
